# 👁️ ARGOS Universal OS v1.4 — Google Colab

> Запуск автономной ИИ-системы прямо в браузере без установки.

**Шаги:**
1. 🔑 Укажи API-ключи в ячейке `Секреты` (или введи вручную)
2. ▶️ Запусти ячейку `Установка` — займёт ~3 минуты
3. ▶️ Запусти ячейку `Старт Аргоса` — откроется чат-интерфейс

---
**Репозиторий:** https://github.com/iliyaqdrwalqu/Argoss

In [ ]:
# ── Проверка окружения ────────────────────────────────────────────────────────
import sys, platform, subprocess

print(f"Python   : {sys.version}")
print(f"Platform : {platform.platform()}")

# Colab даёт ~12 GB RAM и GPU/TPU по запросу.
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                              "--format=csv,noheader"],
                            capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"GPU      : {result.stdout.strip()}")
    else:
        print("GPU      : не обнаружен (CPU режим)")
except Exception:
    print("GPU      : проверка недоступна")

In [ ]:
# ── 🔑 Секреты — заполни или используй Colab Secrets ─────────────────────────
# Рекомендуется: Runtime → Secrets (значок ключа слева)

import os

# Попытка загрузить из Colab Secrets (userdata)
try:
    from google.colab import userdata
    GEMINI_KEY  = userdata.get('GEMINI_API_KEY')  or ''
    TG_TOKEN    = userdata.get('TELEGRAM_BOT_TOKEN') or ''
    TG_USER     = userdata.get('USER_ID')          or ''
    GH_TOKEN    = userdata.get('GITHUB_TOKEN')     or ''
    NET_SECRET  = userdata.get('ARGOS_NETWORK_SECRET') or 'argos_secret_2026'
    print("[✓] Секреты загружены из Colab Secrets")
except Exception:
    # Фallback: вручную
    GEMINI_KEY  = ''   # ← вставь ключ https://aistudio.google.com/app/apikey
    TG_TOKEN    = ''   # ← @BotFather
    TG_USER     = ''   # ← @userinfobot
    GH_TOKEN    = ''   # ← https://github.com/settings/tokens (для сборки APK)
    NET_SECRET  = 'argos_secret_2026'
    print("[!] Colab Secrets недоступны — используются значения из ячейки")

# Записываем .env
env_content = f"""GEMINI_API_KEY={GEMINI_KEY}
TELEGRAM_BOT_TOKEN={TG_TOKEN}
USER_ID={TG_USER}
GITHUB_TOKEN={GH_TOKEN}
ARGOS_NETWORK_SECRET={NET_SECRET}
ARGOS_HOMEOSTASIS=on
ARGOS_CURIOSITY=on
ARGOS_VOICE_DEFAULT=off
ARGOS_TASK_WORKERS=2
ARGOS_OLLAMA_AUTOSTART=on
"""

os.makedirs('/content/Argoss', exist_ok=True)
with open('/content/Argoss/.env', 'w') as f:
    f.write(env_content)
print("[✓] .env записан → /content/Argoss/.env")

In [ ]:
%%bash
# ── Установка системных пакетов ───────────────────────────────────────────────
set -e
echo "[1/5] Системные пакеты..."
apt-get update -qq
apt-get install -y -qq \
    build-essential git zstd \
    libffi-dev python3-dev \
    portaudio19-dev ffmpeg espeak sqlite3 pciutils > /dev/null
echo "[✓] Системные пакеты готовы"

In [ ]:
%%bash
# ── Клонирование / обновление репозитория ─────────────────────────────────────
set -e
echo "[2/5] Репозиторий..."
cd /content
if [ ! -d "SiGtRiP/.git" ]; then
    git clone https://github.com/iliyaqdrwalqu/SiGtRiP.git SiGtRiP
else
    cd SiGtRiP && git pull origin main 2>/dev/null || true
fi
echo "[✓] Репозиторий: /content/SiGtRiP"

In [ ]:
%%bash
# ── Python-зависимости ────────────────────────────────────────────────────────
set -e
echo "[3/5] Python-модули..."
cd /content/Argoss
# Устанавливаем ядро без GUI-зависимостей (customtkinter не нужен в headless)
pip install -q -r requirements.txt 2>&1 | tail -5
echo "[✓] Python-модули готовы"

In [ ]:
%%bash
# ── Ollama (локальный LLM, опционально) ──────────────────────────────────────
echo "[4/5] Ollama (локальный LLM)..."
if [ ! -f "/usr/local/bin/ollama" ]; then
    curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
    echo "[✓] Ollama установлена"
else
    echo "[✓] Ollama уже установлена"
fi

pkill ollama 2>/dev/null || true
sleep 1
nohup ollama serve > /tmp/ollama.log 2>&1 &
echo "[~] Ожидаю порт 11434..."
for i in $(seq 1 20); do
    curl -s http://localhost:11434 > /dev/null 2>&1 && echo "[✓] Ollama запущена (порт 11434)" && break
    sleep 3
done

# Качаем легковесную модель только если Gemini-ключ не задан
if [ -z "$GEMINI_API_KEY" ]; then
    echo "[4/5] Загрузка llama3:8b (может занять несколько минут)..."
    ollama pull llama3:8b 2>&1 | tail -3
else
    echo "[✓] Gemini API задан — Ollama-модель не требуется"
fi

In [ ]:
%%bash
# ── Инициализация Аргоса ──────────────────────────────────────────────────────
set -e
echo "[5/5] Инициализация..."
cd /content/Argoss
python3 genesis.py 2>/dev/null || true

# Краткая проверка структуры
python3 -c "
import os
files = []
for r,d,f in os.walk('.'):
    d[:] = [x for x in d if x not in ['.git','__pycache__','.buildozer']]
    files += [os.path.join(r,ff) for ff in f]
py = [f for f in files if f.endswith('.py')]
print(f'  Файлов всего : {len(files)}')
print(f'  Python-файлов: {len(py)}')
print(f'  Статус       : {\"OK (88+)\" if len(py)>=88 else str(len(py))+\" (ожидается 88+)\"}')
"
echo "[✓] Аргос готов к запуску"

In [ ]:
# ── 🚀 Запуск Аргоса в интерактивном режиме ───────────────────────────────────
# Переключаемся в папку репозитория и запускаем headless-сервер
import os, sys
os.chdir('/content/Argoss')
if '/content/Argoss' not in sys.path:
    sys.path.insert(0, '/content/Argoss')

from dotenv import load_dotenv
load_dotenv('/content/Argoss/.env')
print("[✓] .env загружен")

# Headless-режим (без GUI): запускает Telegram-бот + P2P + Dashboard
# Для запуска через subprocess (фоново):
import subprocess, threading

proc = subprocess.Popen(
    [sys.executable, 'main.py', '--no-gui'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

def _stream_log():
    for line in proc.stdout:
        print(line, end='')

threading.Thread(target=_stream_log, daemon=True).start()
print("[✓] Аргос запущен (PID", proc.pid, ")")
print("[i] Telegram-бот активен. Пиши в бот для взаимодействия.")
print("[i] Для остановки: proc.terminate()")

---
## 📦 Сборка APK (Android)

Сборка APK выполняется через [Buildozer](https://buildozer.readthedocs.io/) — запускай эту ячейку **отдельно**.
Первая сборка занимает ~30-40 минут (скачивает Android SDK/NDK).

In [ ]:
%%bash
# ── APK-сборка через Buildozer ────────────────────────────────────────────────
set -e
cd /content/Argoss

# Зависимости для Buildozer
apt-get install -y -qq \
    openjdk-17-jdk-headless \
    unzip zip wget \
    libstdc++6 zlib1g > /dev/null

pip install -q buildozer cython

# Запуск сборки (debug APK)
buildozer -v android debug 2>&1 | tail -30

echo ""
ls -lh bin/*.apk 2>/dev/null && echo "[✓] APK готов!" || echo "[!] APK не найден"

---
## 📝 Заметки

| Команда | Описание |
|---------|----------|
| `python main.py --no-gui` | Headless (Telegram + P2P) |
| `python main.py --dashboard` | + Веб-панель :8080 |
| `python main.py --shell` | Интерактивная оболочка |
| `python build_exe.py` | Собрать .exe (Windows/Linux) |
| `buildozer android debug` | Собрать APK |
| `docker-compose up -d` | Запустить в Docker |

**Документация:** https://github.com/iliyaqdrwalqu/Argoss#readme